In [39]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load the feature CSVs
features_30_sec = pd.read_csv('Data/features_30_sec.csv')
features_3_sec = pd.read_csv('Data/features_3_sec.csv')

# Check the shape of the DataFrame to ensure it has the correct number of features
print(f'Shape of features_30_sec: {features_30_sec.shape}')  # Should be (n_samples, n_features)

# Extract labels
def extract_label_from_filename(filename):
    return filename.split('.')[0]

features_30_sec['label'] = features_30_sec['filename'].apply(extract_label_from_filename)

# Prepare the features and labels
X = features_30_sec.drop(columns=['filename', 'label'])

# Select the first 57 features (or specify your own feature names)
selected_features = X.columns[:57]  # Adjusted to use the first 57 features
X = X[selected_features]  # Keep only the selected features

y = features_30_sec['label']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Hyperparameter tuning using GridSearchCV
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto'] + [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(SVC(), param_grid, cv=5)
grid_search.fit(X_train_scaled, y_train)

# Best model
best_svm_model = grid_search.best_estimator_

# Evaluate the best model
y_pred = best_svm_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')

# Save the model and scaler
joblib.dump(best_svm_model, 'Data/svm_genre_model.pkl')
joblib.dump(scaler, 'Data/scaler.pkl')



Shape of features_30_sec: (1000, 60)
              precision    recall  f1-score   support

       blues       0.81      0.85      0.83        20
   classical       0.74      0.85      0.79        20
     country       0.85      0.85      0.85        20
       disco       0.65      0.55      0.59        20
      hiphop       0.89      0.80      0.84        20
        jazz       0.77      0.85      0.81        20
       metal       0.94      0.85      0.89        20
         pop       0.70      0.70      0.70        20
      reggae       0.65      0.75      0.70        20
        rock       0.72      0.65      0.68        20

    accuracy                           0.77       200
   macro avg       0.77      0.77      0.77       200
weighted avg       0.77      0.77      0.77       200

Accuracy: 0.77


['Data/scaler.pkl']